In [1]:
from model_api import model
from data import Data
from my_signatures import Signature, Field
import textwrap

/Users/vojtech/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
lamarckian1 = Signature(
    [Field("task_examples", list, "Samples from problem class")],
    [Field("instruction_proposal", str, "Instructions for solving the problem")],
    textwrap.dedent("""\
    Create a general step-by-step instruction to help the user solve a class of problems.
    
    You are a wise advisor with general knowledge about many tasks.
    Look at examples of the problem class under the 'task_examples' field
    and design a tutorial that will guarantee the user's success at solving similar tasks in the future.
    Make sure your instructions are general and apply to all given samples simultaneously.
    
    Use markdown formatting in you final answer to indicate bullet points and whatever else necessary.
    """)
)

lamarckian2 = Signature(
    [Field("task_examples", list, "Samples from problem class")],
    [Field("instruction_proposal", str, "Instructions for solving the problem")],
    textwrap.dedent("""\
    Create a **general** step-by-step instruction to help the user solve a class of problems.
    
    You are a wise advisor with general knowledge about many tasks.
    Look at examples of the problem class under the 'task_examples' field
    and design a tutorial that will guarantee the user's success at solving similar tasks in the future.
    Make sure your instructions are **TRULY GENERAL** and apply to all given samples **simultaneously**.
                    
    Use markdown formatting in you final answer to indicate bullet points and whatever else necessary.
    """)
)


lamarckian3 = Signature(
    [Field("task_examples", list, "Samples from a problem category")],
    [Field("instruction_proposal", str, "Instructions for solving a different problem of the same category")],
    textwrap.dedent("""\
    Create a **general** step-by-step instruction to help the user solve a category of problems.
    
    You are a wise advisor with general knowledge about many tasks.
    Make sure your instructions are **TRULY GENERAL** and apply to all given samples **simultaneously**.
                    
    Follow these steps to make sure your answer is worthy:
        1 - Look at ALL examples in the 'task_examples' field.
        2 - Identify common elements, find the task category.
        3 - Create a step-by-step tutorial that applies to ALL the examples. 
        4 - Look over your tutorial to make sure it is truly general and helpful.
        5 - Write the final step-by-step instruction     
                    
    Use markdown formatting in you final answer to indicate bullet points and whatever else necessary.
    """)
)

lamarckian4 = Signature(
    [Field("task_examples", list, "Samples from a problem category"), Field("focus", list, "Values to focus on")],
    [Field("instruction_proposal", str, "Instructions for solving a different problem of the same category")],
    textwrap.dedent("""\
    Create a **general** step-by-step instruction to help the user solve a category of problems.
                    
    Follow these steps to make sure your answer is worthy:
        1 - Look at ALL examples in the 'task_examples' field.
        2 - Identify common elements, find the task category.
        3 - Create a step-by-step tutorial that applies to ALL the examples. 
        4 - Look over your tutorial to make sure it is truly general and helpful.
        5 - Write the final step-by-step instruction     
    """)
)

lamarckian5 = Signature(
    [Field("task_examples", list, "Samples from a problem class"), Field("focus", list, "Values to focus on")],
    [Field("instruction_proposal", str, "Instructions for solving the problem")],
    textwrap.dedent("""\
    Create a **general** step-by-step instruction to help the user solve a class of problems.
    
    You are a wise advisor with general knowledge about many tasks.
    Look at examples of the problem class under the 'task_examples' field
    and design a tutorial that will guarantee the user's success at solving similar tasks in the future.
    Make sure your instructions are **TRULY GENERAL** and apply to all given samples **simultaneously**.
                    
    Use markdown formatting in you final answer to indicate bullet points and whatever else necessary.
    """)
)

lamarckian6 = Signature(
    [Field("task_examples", list, "Samples from a problem class"), Field("focus", list, "Values to focus on")],
    [Field("prompt_proposal", str, "Instructions for solving the problem")],
    textwrap.dedent("""\
    Craft **general** developer prompt to help an LLM with solving a class of problems.
    
    You are an intelligent instruction induction function capable of advanced reasoning and prompt synthesis.
    Look at examples of the problem class under the 'task_examples' field
    and design a prompt that will guarantee success at solving similar tasks in the future.
    Make sure your instructions are **TRULY GENERAL** and apply to all given samples **simultaneously**.
                    
    Use markdown formatting in you final answer to indicate bullet points and whatever else necessary.
    In the final answer, do not include a title or any additional data, just the prompt.
    """)
)
print(lamarckian3.as_dict())

{'instructions': "Create a **general** step-by-step instruction to help the user solve a category of problems.\n\nYou are a wise advisor with general knowledge about many tasks.\nMake sure your instructions are **TRULY GENERAL** and apply to all given samples **simultaneously**.\n\nFollow these steps to make sure your answer is worthy:\n    1 - Look at ALL examples in the 'task_examples' field.\n    2 - Identify common elements, find the task category.\n    3 - Create a step-by-step tutorial that applies to ALL the examples. \n    4 - Look over your tutorial to make sure it is truly general and helpful.\n    5 - Write the final step-by-step instruction     \n\nUse markdown formatting in you final answer to indicate bullet points and whatever else necessary.\n", 'inputs': {'task_examples': None}, 'outputs': {'instruction_proposal': {'type': "<class 'str'>", 'description': 'Instructions for solving a different problem of the same category'}}}


In [3]:
import random
gsm = Data.from_json("archive/gsm8k-smol.json", float)

with open("datasets/values.csv", "r") as f:
    values = [l.split(',')[0] for l in f.read().split('\n')]

vals = lambda: random.sample(values,3)
vals()

['intellectual rigor', 'adaptability', 'pragmatism']

In [ ]:
gsm_prompts = [model.chain_of_thought(lamarckian6, temp=0.85, task_examples=gsm.train, focus=vals()) for _ in range(10)]


```json
{
"reasoning": "The examples all involve multi-step word problems that require extracting numerical information, performing calculations, and arriving at a final numerical answer. A successful prompt should emphasize a step-by-step approach to problem-solving, focusing on identifying key quantities and operations. The prompt should also explicitly ask the model to show its work, to ensure transparency and facilitate debugging. Furthermore, it should encourage the model to handle percentages and ratios correctly.",
"prompt_proposal": "- You are an expert word problem solver.\n- Your goal is to accurately solve multi-step math problems described in natural language.\n- **Here's how you should approach each problem:**\n  - **Step 1: Identify the key quantities and units.** What numbers are given, and what are we trying to find?\n  - **Step 2: Break down the problem into smaller, manageable steps.** What operations (addition, subtraction, multiplication, division, percentages, rati

TypeError: 'NoneType' object is not subscriptable

In [8]:
for r,p in gsm_prompts:
    if p:
        print(r, '\n', p["prompt_proposal"])

The problems all involve multi-step arithmetic word problems. They require identifying the relevant numbers and operations, performing calculations in the correct order, and arriving at the final numerical answer. A good prompt should focus on breaking down the problem into smaller steps, explicitly performing calculations, and clearly stating the final answer. The prompt should also emphasize the importance of understanding the context of each number given in the problem. The 'focus' keywords suggest we should make the prompt useful and able to deliver the correct answer in a reliable way. 
 - You are a world-class mathematician and problem solver.\n- Your task is to solve multi-step arithmetic word problems.\n- **Read the problem carefully.** Identify the key quantities and what the question is asking.\n- **Break down the problem into smaller, manageable steps.** What calculations need to be done, and in what order?\n- **Perform each calculation accurately.** Show your work step-by-s

In [9]:
reflective = Signature(
    [
        Field("original_prompt", str, ""),
        Field("task_question", str, ""),
        Field("solution", str, "")
    ],
    [Field("original_prompt_critique", str, ""), Field("prompt_proposal", str, "")],
    textwrap.dedent("""\
    Improve a prompt for an LLM.
    
    You are an intelligent reflection function capable of advanced reasoning and prompt synthesis.
    Follow these steps to craft a better prompt:
    - Analyze the original prompt and its suboptimal performance on a task sample.
    - Find failure points in the solution and cross-reference to identify weaknesses in the prompt.
    - Think of a critique that captures your findings
    - Apply your critique to *slightly* alter the original prompt to improve it.
    Your improved prompt should still be **widely applicable and generic**.

    Maintain the same formatting as in the original prompt.  
    In the final answer, do not include a title or any additional data, just the prompt.
    """)
)

solve_sig = Signature.from_str(f"task: str (task to be solved) -> solution: int ()")
solve_module = lambda task: model.chain_of_thought(solve_sig, temp=0.0, task=task)

In [10]:
p = gsm_prompts[0][1]["prompt_proposal"]
task = "Let $(X, A, m)$ be a probability space, and let $E_n$ be measurable sets such that every $x \in X$ belongs to exactly 3 of the sets $E_n$. Compute the sum $\sum m(E_n)$. Provide your answer as a single value."
reasoning, ans = solve_module(task=task)

<>:2: SyntaxWarning: invalid escape sequence '\i'
<>:2: SyntaxWarning: invalid escape sequence '\i'
/var/folders/r3/_sqxxd5n0bjd_f5bjc4w0dch0000gn/T/ipykernel_36672/3995403931.py:2: SyntaxWarning: invalid escape sequence '\i'
  task = "Let $(X, A, m)$ be a probability space, and let $E_n$ be measurable sets such that every $x \in X$ belongs to exactly 3 of the sets $E_n$. Compute the sum $\sum m(E_n)$. Provide your answer as a single value."


In [11]:
reasoning

"Let $X$ be a set and $A$ be a sigma-algebra on $X$. Let $m$ be a measure on $(X, A)$. We are given a sequence of measurable sets $E_n$ such that every $x \\in X$ belongs to exactly 3 of the sets $E_n$. We want to compute $\\sum_n m(E_n)$.\\n\\nConsider the set $S = \\{(x, n) : x \\in E_n\\}$. We can also write $S = \\bigcup_n (E_n \\times \\{n\\})$.\\nSince every $x \\in X$ belongs to exactly 3 of the sets $E_n$, for each $x$, there are exactly 3 pairs $(x, n)$ in $S$. Therefore, the cardinality of the set $S$ is $3|X|$.\\n\\nNow, we can compute the measure of $S$ in two ways. First, we can sum over $n$: \\n$$m(S) = \\sum_n m(E_n \\times \\{n\\})$$ \\nSince $E_n \\times \\{n\\}$ are disjoint sets, we have \\n$$m(S) = \\sum_n m(E_n \\times \\{n\\}) = \\sum_n m(E_n) m(\\{n\\}) = \\sum_n m(E_n) \\cdot 0 = 0$$ \\nThis is incorrect, as we are assuming that $m(\\{n\\}) = 0$. However, we can consider the measure $m \\times \\delta_n$ where $\\delta_n$ is the Dirac delta measure at $n$. Then 

In [12]:
refl_r, relf_a = model.chain_of_thought(reflective, temp=0.85, original_prompt=p, task_question=task, solution=reasoning)

In [13]:
relf_a

{'original_prompt_critique': "The prompt is too focused on arithmetic word problems and includes irrelevant instructions (like counting problems). It's overly detailed and verbose, possibly distracting the LLM. The emphasis on units and whole numbers is not applicable to the provided problem.",
 'prompt_proposal': '- You are a highly skilled mathematician.\\n- Solve the given problem, showing your work and explaining each step.\\n- Express your final answer clearly and concisely, using mathematical notation where appropriate.\\n- If the solution depends on unspecified variables, state your assumptions.'}